# Cómo obtener propiedades termodinámicas de los diferentes paquetes ya existentes

- thermochem
    - iapws
    - janaf
    - burcat
- iapws
- coolprop

## thermochem - iapws
iapws permite evaluar, solo para water:
    - h a partir de T y P
    - T de saturación a partir de P
    - P de saturación a partir de T
    - T a partir de h y P

Basado en International Association for the Properties of Water and
Steam. Lucerne, Switzerland. August 2007. 

Tiene los coeficientes de las funciones cargados en el propio script, no necesita acceder a otra base de datos.

In [1]:
import thermochem.iapws as iapws

w = iapws.Water() #genera el objeto agua

T = 373.15 # temperatura en K

P = 100000/100 # presión en Pa

h = w.h(P, T)  # entalpía en kJ/kg

print(f"Enthalpy {h} kJ/kg at {T} K and {P} Pa")

Enthalpy 2688.536040687925 kJ/kg at 373.15 K and 1000.0 Pa


c:\Users\jazmi\Documents\Balseiro\Pythoncientifico\CursoPythonIB_proy_termo\.venv\Lib\site-packages\thermochem\units.py:391: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
  * :math:`\\frac{kg}{s\ m^2}` (default)



No devuelve en qué estado de agregación está.

Si P < Psat, devuelve la entalpía de VSC

Si P >= Psat, devuelve la entalpía de LC o Lsat

Una aproximación a la h de Vsat se puede obtener evaluando en Psat-0.0001

In [2]:
hv_sat = w.h(w.psat(T)-0.0001, T)
print(f"Enthalpy {hv_sat} kJ/kg at {T} K and {w.psat(T)-0.0001} Pa")

Enthalpy 2675.572017206459 kJ/kg at 373.15 K and 101417.97782131012 Pa


In [3]:
Psat = w.psat(T)  # presión de saturación en Pa

Tsat = w.Tsat(P)  # temperatura de saturación en K

print(f"Saturation pressure of water at {T} K: {Psat} Pa")
print(f"Saturation temperature of water at {P} Pa: {Tsat} K")

Saturation pressure of water at 373.15 K: 101417.97792131013 Pa
Saturation temperature of water at 1000.0 Pa: 280.11963241256103 K


In [4]:
Temp = w.T_ph(P/1000000, h)[1] #T(K) a partir de P (MPa) y h (kJ/kg). Para P < 4 MPa.

print(f"Temperature {Temp:.2f} K at {P/1000000} MPa and {h:.2f} kJ/kg")

Temperature 373.15 K at 0.001 MPa and 2688.54 kJ/kg


In [5]:
print(h)

2688.536040687925


In [6]:
import urllib.request
import thermochem.janaf as Janaf

# Configurar un opener global para urllib.request con un User-Agent
# personalizado

opener = urllib.request.build_opener()
opener.addheaders = [
    ("User-Agent", "Mozilla/5.0 (compatible; Python-thermochem-client/1.0)")
]
urllib.request.install_opener(opener)

In [7]:
db = Janaf.Janafdb()  # Crear el objeto de la base de datos

# Buscar resultados compatibles con H2O en la base de datos
search_results = db.search("H2O")

# Estos serán los keys a utilizar para extraer los datos.
print(search_results)

     formula                 name phase filename
534     CH2O         Formaldehyde     g    C-061
1213     H2O                Water     l    H-063
1214     H2O                Water     g    H-064
1215     H2O         Water, 1 Bar   l,g    H-065
1216     H2O        Water, 10 Bar   l,g    H-066
1217     H2O       Water, 100 Bar   l,g    H-067
1218     H2O       Water, 500 Bar    fl    H-068
1219     H2O      Water, 5000 Bar    fl    H-069
1220    H2O2    Hydrogen Peroxide     g    H-070
1221  H2O2Sr  Strontium Hydroxide    cr    H-071
1222  H2O2Sr  Strontium Hydroxide     l    H-072
1223  H2O2Sr  Strontium Hydroxide  cr,l    H-073
1224  H2O2Sr  Strontium Hydroxide     g    H-074
1225   H2O4S        Sulfuric Acid  cr,l    H-075
1226   H2O4S        Sulfuric Acid     g    H-076
1227   H2O4W        Tungstic Acid    cr    H-077
1228   H2O4W        Tungstic Acid     g    H-078


In [8]:
# Extraer los datos, de acuerdo a la formula y fase elegida.
datos = db.getphasedata(
    formula="H2O",
    name="Water",
    phase="l",
    filename="H-063",
    cache=False
    )

print(datos.S([298, 300, 302])/18)        # Entropy in J/g/K

print("falta imprimir tabla")

[3.8839371  3.912      3.93901667]
falta imprimir tabla


c:\Users\jazmi\Documents\Balseiro\Pythoncientifico\CursoPythonIB_proy_termo\.venv\Lib\site-packages\thermochem\janaf.py:80: ParserWarning: Skipping line 12: Expected 8 fields in line 12, saw 9. Error could possibly be due to quotes being ignored when a multi-char delimiter is used.

  data = pd.read_csv(


- IAPWS

Solo para Water

Al igual que el código cargado en thermochem, este paquete está basado en las funciones:

IAPWS, Revised Release on the IAPWS Industrial Formulation 1997 for the
Thermodynamic Properties of Water and Steam August 2007,
http://www.iapws.org/relguide/IF97-Rev.html

Pero, a diferencia del anterior, incluye actualizaciones y la implementación es mucho más amplia.

Indica en qué fase se encuentra.

In [9]:
from iapws import IAPWS97

Pressure = 0.1  # MPa

Temperature = 500  # K

# Vapor sobrecalentado (VSC)
vsc = IAPWS97(P=Pressure, T=Temperature) 

print((f"fase: {vsc.phase}; entalpía: {vsc.h:.2f} kJ/kg; "
       f"entropía: {vsc.s:.3f} kJ/kg/K"))

fase: Vapour; entalpía: 2928.59 kJ/kg; entropía: 7.945 kJ/kg/K


In [10]:

# Mezcla líquido-vapor en equilibrio

Pressure = 0.1  # MPa

titulo = 0.5  # calidad de vapor

mezcla = IAPWS97(P=Pressure, x=titulo)

print((f"fase: {mezcla.phase}; entalpía: {mezcla.h:.2f} kJ/kg; "
       f"entropía: {mezcla.s:.2f} kJ/kg/K"))

print((f"hf: {mezcla.Liquid.h:.2f} kJ/kg; hg: {mezcla.Vapor.h:.2f} kJ/kg; \n"
       f"sf: {mezcla.Liquid.s:.4f} kJ/(kg K); "
       f"sg: {mezcla.Vapor.s:.4f} kJ/(kg K)"))

#Se puede evaluar x, a partir de P y h, o P y s.

mezcla_2 = IAPWS97(P = Pressure, h = 1500) #líquido saturado

print((f"fase: {mezcla_2.phase}; título: {mezcla_2.x:.2f}; "
       f"entropía: {mezcla_2.s:.4f} kJ/kg/K"))

fase: Two phases; entalpía: 1546.19 kJ/kg; entropía: 4.33 kJ/kg/K
hf: 417.44 kJ/kg; hg: 2674.95 kJ/kg; 
sf: 1.3026 kJ/(kg K); sg: 7.3588 kJ/(kg K)
fase: Two phases; título: 0.48; entropía: 4.2068 kJ/kg/K


In [11]:
# Líquido comprimido

Pressure = 10  # MPa

Temperature = 300  # K

liq_comp = IAPWS97(P=Pressure, T=Temperature)

print((f"fase: {liq_comp.phase}; entalpía: {liq_comp.h:.2f} kJ/kg; "
      f"entropía: {liq_comp.s:.2f} kJ/kg/K"))

fase: Liquid; entalpía: 121.75 kJ/kg; entropía: 0.39 kJ/kg/K


## Coolprop

CoolProp puede calcular propiedades termodinámicas —como densidad, entalpía, entropía, energía interna y calores específicos— y propiedades de transporte, como viscosidad y conductividad térmica. También permite consultar estados de líquido, vapor, saturación y supercríticos dentro del rango de validez del modelo.

"Water" usa el backend HEOS. Para el agua, su ecuación de estado se basa en la formulación IAPWS-95, publicada por Wagner y Pruß en 2002 (referencia).

"IF97::Water" usa IAPWS-IF97, la formulación industrial para agua y vapor; está pensada para cálculos más rápidos en aplicaciones industriales. Sus límites de uso son distintos, y su precisión es menor cerca del punto crítico.

In [12]:
import CoolProp.CoolProp as CP

P = 1e5       # Pa
T = 373.15    # K

h = CP.PropsSI("Hmass", "P", P, "T", T, "Water")  # J/kg
rho = CP.PropsSI("Dmass", "P", P, "T", T, "Water") # kg/m³
cp = CP.PropsSI("Cpmass", "P", P, "T", T, "Water") # J/(kg K)

print((f"Enthalpy: {h/1000:.2f} kJ/kg, Density: {rho:.2f} kg/m³, \n"
       f"Specific Heat Capacity: {cp/1000:.2f} kJ/(kg K) at {T} K and {P} Pa"))

Enthalpy: 2675.77 kJ/kg, Density: 0.59 kg/m³, 
Specific Heat Capacity: 2.08 kJ/(kg K) at 373.15 K and 100000.0 Pa


In [13]:
# Las propiedades de líquido y vapor saturado se obtienen, para una dada
# presión, fijando Q = 0 y Q = 1, respectivamente.

h_liq = CP.PropsSI("Hmass", "P", P, "Q", 0, "Water")
h_vap = CP.PropsSI("Hmass", "P", P, "Q", 1, "Water")

print((f"Enthalpy of saturated liquid: {h_liq/1000:.2f} kJ/kg;\n"
      f"Enthalpy of saturated vapor: {h_vap/1000:.2f} kJ/kg;\nat {P} Pa"))


Enthalpy of saturated liquid: 417.50 kJ/kg;
Enthalpy of saturated vapor: 2674.95 kJ/kg;
at 100000.0 Pa


In [14]:
from CoolProp.CoolProp import PhaseSI

fase = PhaseSI("P", 1e5, "T", 300, "Water")
print(fase)  # liquid

liquid


In [15]:
# Se pueden consultar otros fluidos disponibles

fluidos = CP.get_global_param_string("FluidsList").split(",")

for fluido in fluidos:
    print(fluido)

1-Butene
Acetone
Air
Ammonia
Argon
Benzene
CarbonDioxide
CarbonMonoxide
CarbonylSulfide
Chlorine
cis-2-Butene
CycloHexane
Cyclopentane
CycloPropane
D4
D5
D6
Deuterium
Dichloroethane
DiethylEther
DimethylCarbonate
DimethylEther
Ethane
Ethanol
EthylBenzene
Ethylene
EthyleneOxide
Fluorine
HeavyWater
Helium
HFE143m
Hydrogen
HydrogenChloride
HydrogenSulfide
IsoButane
IsoButene
Isohexane
Isopentane
Krypton
m-Xylene
MD2M
MD3M
MD4M
MDM
Methane
Methanol
MethylLinoleate
MethylLinolenate
MethylOleate
MethylPalmitate
MethylStearate
MM
n-Butane
n-Decane
n-Dodecane
n-Heptane
n-Hexane
n-Nonane
n-Octane
n-Pentane
n-Perfluorobutane
n-Perfluorohexane
n-Perfluoropentane
n-Propane
n-Undecane
Neon
Neopentane
Nitrogen
NitrousOxide
Novec649
o-Xylene
OrthoDeuterium
OrthoHydrogen
Oxygen
p-Xylene
ParaDeuterium
ParaHydrogen
Propylene
PropyleneGlycol
Propyne
R11
R1123
R113
R1130(E)
R1132(E)
R114
R115
R116
R12
R1224YDZ
R123
R1233zd(E)
R1234yf
R1234ze(E)
R1234ze(Z)
R124
R1243zf
R125
R13
R1336mzz(E)
R1336mzz(Z)
R134